# Step 1 — Colab Ortam Kurulumu

**Amaç:** Google Drive'ı bağla, klasör yapısını oluştur, kütüphaneleri kur, GPU'yu doğrula, lokal code zip'ini Drive'a aç.

**Önce yap:** Runtime → Change runtime type → **GPU (T4)** seç.

**Önce yüklemen gereken:** `Prostate_MRI_Project_code.zip` dosyasını Drive'da `MyDrive/Prostate_MRI_Project/` altına yükle.

## 1.1 — GPU kontrolü

In [ ]:
!nvidia-smi

Beklenen: T4 / P100 / V100 GPU, 16 GB VRAM. Yoksa Runtime → Change runtime type → GPU.

## 1.2 — Google Drive'ı bağla

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1.3 — Proje yolları

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/Prostate_MRI_Project')
CODE_DIR     = PROJECT_ROOT / 'code'
INPUT_DIR    = PROJECT_ROOT / 'input'
IMAGES_DIR   = INPUT_DIR / 'images'
LABELS_DIR   = INPUT_DIR / 'picai_labels'
WORKDIR      = PROJECT_ROOT / 'workdir'
OUTPUT_DIR   = PROJECT_ROOT / 'output'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'

# Klasör yapısını oluştur
for d in [CODE_DIR, INPUT_DIR, IMAGES_DIR, LABELS_DIR, WORKDIR, OUTPUT_DIR, NOTEBOOKS_DIR,
          OUTPUT_DIR / 'anatomy_predictions',
          OUTPUT_DIR / 'lesion_predictions',
          OUTPUT_DIR / 'figures',
          OUTPUT_DIR / 'metrics',
          WORKDIR / 'nnUNet_raw_data',
          WORKDIR / 'nnUNet_preprocessed',
          WORKDIR / 'results']:
    d.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Var olan içerik:')
!ls -la "{PROJECT_ROOT}"

## 1.4 — Lokal kod zip'ini Drive'da aç

`Prostate_MRI_Project_code.zip` dosyasını Drive'a `MyDrive/Prostate_MRI_Project/` altına yüklediysen, aç:

In [ ]:
import zipfile

zip_path = PROJECT_ROOT / 'Prostate_MRI_Project_code.zip'

if zip_path.exists():
    print(f'Zip bulundu: {zip_path}')
    print('Açılıyor...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(CODE_DIR)
    print('Tamam!')
    !ls -la "{CODE_DIR}"
else:
    print(f'UYARI: {zip_path} bulunamadı.')
    print('Lokal Prostate_MRI_Project klasörünü zip\'leyip Drive\'a yükle.')

## 1.5 — picai_labels'ı input/ altına kopyala

PI-CAI baseline scriptleri labels'ı `/input/picai_labels/` altında bekler.

In [ ]:
import shutil

src_labels = CODE_DIR / 'picai_labels-main'
if src_labels.exists():
    if not (LABELS_DIR / 'anatomical_delineations').exists():
        # Drive'a klasör kopyalamak yavaş — sadece sembolik benzeri yapacağız
        for item in src_labels.iterdir():
            dst = LABELS_DIR / item.name
            if item.is_dir() and not dst.exists():
                shutil.copytree(item, dst)
            elif item.is_file() and not dst.exists():
                shutil.copy(item, dst)
        print('Labels kopyalandı.')
    else:
        print('Labels zaten kopyalanmış.')
    !ls "{LABELS_DIR}"
else:
    print('UYARI: picai_labels-main bulunamadı, zip\'i açtın mı?')

## 1.6 — Kütüphaneleri kur

Colab'da PyTorch zaten var. MONAI, SimpleITK, picai modüllerini ekliyoruz.

In [ ]:
# Temel medical imaging stack
!pip install -q "monai[all]==1.3.2" SimpleITK nibabel itk pydicom

# PI-CAI özel modülleri
!pip install -q picai-eval picai-prep

# Görselleştirme
!pip install -q matplotlib seaborn plotly

# picai_baseline'ı kod klasöründen kur
baseline_dir = CODE_DIR / 'picai_baseline-main'
if baseline_dir.exists():
    !pip install -q -e "{baseline_dir}"
    print('picai_baseline kuruldu.')
else:
    print('UYARI: picai_baseline-main bulunamadı.')

## 1.7 — Kurulum doğrulama

In [ ]:
import torch
import monai
import SimpleITK as sitk
import numpy as np

print(f'PyTorch:       {torch.__version__}')
print(f'CUDA available:{torch.cuda.is_available()}')
print(f'CUDA device:   {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK"}')
print(f'VRAM:          {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')
print(f'MONAI:         {monai.__version__}')
print(f'SimpleITK:     {sitk.Version_VersionString()}')

try:
    import picai_baseline, picai_eval, picai_prep
    print(f'picai_baseline: OK')
    print(f'picai_eval:     OK')
    print(f'picai_prep:     OK')
except ImportError as e:
    print(f'UYARI: {e}')

## 1.8 — Disk durumu

In [ ]:
!df -h /content /content/drive/MyDrive 2>/dev/null

## 1.9 — Hızlı annotation envanteri

PI-CAI labels'ta ne kadar veri/hasta var bakalım:

In [ ]:
import pandas as pd

marksheet_path = LABELS_DIR / 'clinical_information' / 'marksheet.csv'
if marksheet_path.exists():
    df = pd.read_csv(marksheet_path)
    print(f'Toplam hasta: {len(df)}')
    print(f'Sütunlar: {list(df.columns)}')
    print('\nİlk 5 satır:')
    display(df.head())
    if 'case_ISUP' in df.columns:
        print('\nISUP dağılımı:')
        print(df['case_ISUP'].value_counts(dropna=False).sort_index())
else:
    print(f'marksheet.csv bulunamadı: {marksheet_path}')

In [ ]:
# Lezyon ve anatomi label sayıları
lesion_human = LABELS_DIR / 'csPCa_lesion_delineations' / 'human_expert'
if lesion_human.exists():
    # Birden çok versiyon olabilir (resampled, lesions_ai, vs)
    for sub in lesion_human.iterdir():
        if sub.is_dir():
            files = list(sub.glob('*.nii.gz'))
            print(f'  {sub.name}: {len(files)} dosya')

anat_zonal = LABELS_DIR / 'anatomical_delineations' / 'zonal_pz_tz' / 'AI'
if anat_zonal.exists():
    for sub in anat_zonal.iterdir():
        if sub.is_dir():
            files = list(sub.glob('*.nii.gz'))
            print(f'  zonal/{sub.name}: {len(files)} dosya')

---

## ✅ Step 1 Bitti

**Bir sonraki adım:** `step2_download_data.ipynb` — PI-CAI fold 0 imaging verisini Zenodo'dan Drive'a indirme.

**Hatırlatma:** Bu notebook'u tamamen çalıştırdıktan sonra Drive'da `MyDrive/Prostate_MRI_Project/notebooks/` altına kaydet.